# Week 6 Lab: Model Comparisons

This lab is about the *tools* for comparing models, in two flavors:

- **Part 1 (Regression):** compare four delay-discounting models -- exponential,
  hyperbolic, Green & Myerson's hyperboloid, and Rachlin's hyperboloid -- using
  R-squared, MAE, RMSE, AIC, BIC, and AICc.
- **Part 2 (Classification):** compare logistic regression, a decision tree, a
  random forest, and an SVM at predicting challenging behavior, using accuracy,
  precision, recall, F1, MCC, and ROC-AUC.

Information criteria (AIC/BIC/AICc) matter when models differ in parameter count:
they penalize complexity so the comparison is fair.

## Setup

Import everything you need: `scipy.optimize` (fitting), `sklearn.metrics`, and the four classifier families from scikit-learn.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.optimize import differential_evolution

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, matthews_corrcoef, roc_auc_score, roc_curve)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

# Part 1: Discounting Model Comparison (Regression)

## Task 1: Load the discounting data

Each row is one participant/commodity with indifference points at 7 delays plus the `Amount` and `Commodity`.

In [ ]:
raw_data = pd.read_csv('participant_discounting_data.csv')
raw_data.head()

## Task 2: Define the four discounting models

Each returns predicted value as a proportion of the amount (so indifference points are normalized to 0-1).

- Exponential: $V = e^{-kD}$
- Hyperbolic (Mazur): $V = 1/(1+kD)$
- Green-Myerson: $V = 1/(1+kD)^s$
- Rachlin: $V = 1/(1+kD^s)$

In [ ]:
def exponential_model(delay, k):
    return np.exp(-k * delay)

def hyperbolic_model(delay, k):
    return 1 / (1 + k * delay)

def green_myerson_hyperboloid(delay, k, s):
    return 1 / ((1 + k * delay) ** s)

def rachlin_hyperboloid(delay, k, s):
    return 1 / (1 + k * (delay ** s))

## Task 3: Helpers to fit a model and compute fit metrics

`fit_model_robust` uses global optimization (`differential_evolution`) to minimize squared error. `calculate_fit_metrics` returns R-squared, MAE, RMSE plus the information criteria AIC, BIC, and AICc (small-sample corrected).

In [ ]:
def prepare_data_for_fitting(raw_data, participant_id, commodity):
    row = raw_data[(raw_data['participant_id'] == participant_id) &
                   (raw_data['Commodity'] == commodity)].iloc[0]
    delay_cols = [c for c in raw_data.columns if c.startswith('delay_')]
    delays = np.array([float(c.split('_')[1]) for c in delay_cols])
    indiff = row[delay_cols].values.astype(float) / row['Amount']   # normalize to 0-1
    return delays, indiff

def fit_model_robust(model_func, delays, indiff, bounds):
    def objective(params):
        try:
            return np.sum((indiff - model_func(delays, *params)) ** 2)
        except Exception:
            return 1e10
    result = differential_evolution(objective, bounds, seed=0, maxiter=200, tol=1e-6)
    params = result.x
    predictions = model_func(delays, *params)
    return params, predictions

def calculate_fit_metrics(observed, predicted, n_params, n_obs):
    residuals = observed - predicted
    ss_res = np.sum(residuals ** 2)
    ss_tot = np.sum((observed - np.mean(observed)) ** 2)
    r_squared = 1 - (ss_res / ss_tot) if ss_tot > 0 else 0
    mae = np.mean(np.abs(residuals))
    rmse = np.sqrt(np.mean(residuals ** 2))
    mse = ss_res / n_obs
    log_lik = -0.5 * n_obs * (np.log(2 * np.pi * mse) + 1) if mse > 0 else 0
    aic = 2 * n_params - 2 * log_lik
    bic = np.log(n_obs) * n_params - 2 * log_lik
    aicc = aic + (2 * n_params * (n_params + 1)) / (n_obs - n_params - 1) \
        if n_obs - n_params - 1 > 0 else np.inf
    return {'R2': r_squared, 'MAE': mae, 'RMSE': rmse, 'AIC': aic, 'BIC': bic, 'AICc': aicc}

MODELS = {
    'Exponential':   {'func': exponential_model,        'bounds': [(1e-6, 10)],            'n_params': 1},
    'Hyperbolic':    {'func': hyperbolic_model,         'bounds': [(1e-6, 10)],            'n_params': 1},
    'Green-Myerson': {'func': green_myerson_hyperboloid,'bounds': [(1e-6, 10), (0.1, 5)],  'n_params': 2},
    'Rachlin':       {'func': rachlin_hyperboloid,      'bounds': [(1e-6, 10), (0.1, 5)],  'n_params': 2},
}

## Task 4: Fit every model to every participant

Loop over participants, fit all four models, and collect the metrics into a tidy summary DataFrame. (Fitting all participants on 7 points each takes a moment.)

In [ ]:
summary_rows = []
for pid in raw_data['participant_id'].unique():
    for comm in raw_data[raw_data['participant_id'] == pid]['Commodity'].unique():
        delays, indiff = prepare_data_for_fitting(raw_data, pid, comm)
        for name, m in MODELS.items():
            params, preds = fit_model_robust(m['func'], delays, indiff, m['bounds'])
            metrics = calculate_fit_metrics(indiff, preds, m['n_params'], len(indiff))
            summary_rows.append({'Participant': pid, 'Commodity': comm, 'Model': name, **metrics})

summary_df = pd.DataFrame(summary_rows)
summary_df.head(8)

## Task 5: Compare the models across metrics

For each fit metric, show its distribution by model so you can see which model wins on which metric. Note how the information criteria treat the 1- vs 2-parameter models differently.

In [ ]:
mean_by_model = summary_df.groupby('Model')[['R2', 'MAE', 'RMSE', 'AIC', 'BIC', 'AICc']].mean()
print(mean_by_model.round(4))

metrics_to_plot = ['R2', 'MAE', 'RMSE', 'AIC', 'BIC', 'AICc']
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, metric in zip(axes.ravel(), metrics_to_plot):
    sns.boxplot(data=summary_df, x='Model', y=metric, ax=ax)
    ax.set_title(metric)
    ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

# Part 2: Challenging Behavior Prediction (Classification)

## Task 6: Load and prepare the data

Load `challenging_behavior_data.csv`, split features/label, make a train/test split, and standardize features (needed by logistic regression and the SVM). The SVM is expensive on large data, so we sample a subset for training speed.

In [ ]:
clf_data = pd.read_csv('challenging_behavior_data.csv')

# RBF-SVM training cost grows steeply with n; sample for a tractable demo.
clf_data = clf_data.sample(n=5000, random_state=42).reset_index(drop=True)

X = clf_data.drop('ChallengingBehavior', axis=1)
y = clf_data['ChallengingBehavior']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print('train:', X_train.shape, ' test:', X_test.shape)

## Task 7: Fit four classifiers and score them

Fit logistic regression, a decision tree, a random forest, and an RBF SVM. For each, compute accuracy, precision, recall, F1, MCC, and ROC-AUC, and add its ROC curve to a shared plot. Scale-sensitive models (logistic, SVM) use the scaled features.

In [ ]:
classifiers = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'SVM (RBF)': SVC(probability=True, random_state=42),
}

results = []
plt.figure(figsize=(9, 7))
for name, model in classifiers.items():
    if 'SVM' in name or 'Logistic' in name:
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        y_proba = model.predict_proba(X_test_scaled)[:, 1]
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1]

    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1 Score': f1_score(y_test, y_pred),
        'MCC': matthews_corrcoef(y_test, y_pred),
        'ROC AUC': roc_auc_score(y_test, y_proba),
    })
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {results[-1]['ROC AUC']:.2f})")

plt.plot([0, 1], [0, 1], 'k--', label='Random Guess')
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('ROC Curves'); plt.legend(); plt.grid(True); plt.tight_layout()
plt.show()

## Task 8: Results table and interpretation

Put the classification metrics in one table. Which model would you choose, and on which metric -- and why might accuracy alone be misleading if challenging behavior is rare?

In [ ]:
results_df = pd.DataFrame(results).set_index('Model').round(3)
print(results_df)

## Wrap-up

Across both parts: when does a more complex model (extra parameter, ensemble) actually earn its complexity? Tie your answer to what the information criteria showed in Part 1 and to the precision/recall trade-off in Part 2.